# Ch 34 (검증) — 한국어 diffusion, 80/10/10 수정판

순진한 100% [MASK] diffusion은 한국어에서 유니그램 붕괴(acc 0.08). BERT의 80/10/10 마스킹 트릭 + plain CE 이식으로 해결(검증 acc 0.467). 생성까지 확인.

In [ ]:
%pip install -q -U transformers tokenizers datasets accelerate

In [ ]:
import math, time, torch
import torch.nn.functional as F
from datasets import load_dataset, Dataset
SEED=42; torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
USE_FP16 = torch.cuda.is_available()
print("device", device, "| fp16", USE_FP16)

## 1. 한국어 TinyStories 복원 (Ch 26과 동일)

In [ ]:
EOT="<|endoftext|>"; N_TRAIN,N_VAL,MAXL=50_000,500,1_500_000
def rebuild(split,n,maxl):
    stories,buf=[],[]
    for i,ex in enumerate(load_dataset("g0ster/TinyStories-Korean",split=split,streaming=True)):
        if i>=maxl or len(stories)>=n: break
        line=(ex["text"] or "").strip()
        if line==EOT:
            s=" ".join(buf).strip()
            if s: stories.append(s)
            buf=[]
        elif line: buf.append(line)
    if buf and len(stories)<n:
        s=" ".join(buf).strip()
        if s: stories.append(s)
    return stories[:n]
raw_train=Dataset.from_dict({"text":rebuild("train",N_TRAIN,MAXL)})
raw_val=Dataset.from_dict({"text":rebuild("validation",N_VAL,50_000)})
print("stories", len(raw_train), len(raw_val))
print(raw_train[0]["text"][:120])

## 2. BPE 4000 + initial_alphabet + [MASK]

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from transformers import PreTrainedTokenizerFast
VOCAB=4000
def corpus_iter(bs=1000):
    for i in range(0,len(raw_train),bs): yield raw_train[i:i+bs]["text"]
_tk=Tokenizer(models.BPE(unk_token="[UNK]"))
_tk.pre_tokenizer=pre_tokenizers.ByteLevel(add_prefix_space=False)
_tk.decoder=decoders.ByteLevel()
_tk.train_from_iterator(corpus_iter(), trainer=trainers.BpeTrainer(
    vocab_size=VOCAB, special_tokens=["[PAD]","[UNK]","[MASK]"],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet()))
tokenizer=PreTrainedTokenizerFast(tokenizer_object=_tk, pad_token="[PAD]", unk_token="[UNK]", mask_token="[MASK]")
print("vocab", tokenizer.vocab_size, "mask_id", tokenizer.mask_token_id)

## 3. 토큰화 + group

In [ ]:
BLOCK=128
tt=raw_train.map(lambda b: tokenizer(b["text"],add_special_tokens=False), batched=True, remove_columns=raw_train.column_names)
tv=raw_val.map(lambda b: tokenizer(b["text"],add_special_tokens=False), batched=True, remove_columns=raw_val.column_names)
def group(b):
    cat=sum(b["input_ids"],[]); n=(len(cat)//BLOCK)*BLOCK
    return {"input_ids":[cat[i:i+BLOCK] for i in range(0,n,BLOCK)]}
lm_train=tt.map(group,batched=True,remove_columns=tt.column_names)
lm_val=tv.map(group,batched=True,remove_columns=tv.column_names)
print("chunks", len(lm_train), len(lm_val))

## 4. ★수정 — diffusion 콜레이터에 80/10/10 (순진한 100% [MASK] 대신)

가변 마스킹률 t는 유지(생성용)하되, 선택된 자리에 80% [MASK] / 10% 랜덤 / 10% 원본유지. 이게 모델이 [MASK]→유니그램 지름길로 새는 걸 막는다.

In [ ]:
N_SPECIAL=3
class DiffMLMCollator:
    def __init__(self, tok, eps=0.05, tmax=1.0, seed=SEED):
        self.mask_id=tok.mask_token_id; self.vocab=tok.vocab_size
        self.eps=eps; self.tmax=tmax; self.gen=torch.Generator().manual_seed(seed)
    def __call__(self, ex):
        ids=torch.tensor([e["input_ids"] for e in ex], dtype=torch.long)
        B,L=ids.shape
        t=torch.rand(B,generator=self.gen)*(self.tmax-self.eps)+self.eps
        sel=torch.rand(B,L,generator=self.gen)<t.unsqueeze(1)
        no=~sel.any(1)
        if no.any():
            j=torch.randint(0,L,(int(no.sum()),),generator=self.gen); sel[no,j]=True
        labels=ids.clone(); labels[~sel]=-100
        inp=ids.clone()
        r=torch.rand(B,L,generator=self.gen)
        inp[sel&(r<0.8)]=self.mask_id                                    # 80% [MASK]
        rp=sel&(r>=0.8)&(r<0.9); nr=int(rp.sum())                        # 10% 랜덤
        if nr: inp[rp]=torch.randint(N_SPECIAL,self.vocab,(nr,),generator=self.gen)
        # 10% 원본 유지
        return {"input_ids":inp,"attention_mask":torch.ones(B,L,dtype=torch.long),"labels":labels}
coll=DiffMLMCollator(tokenizer)

## 5. 작은 모델 (256/4L) — 용량 아닌 마스킹이 문제였음

In [ ]:
from transformers import BertConfig, BertForMaskedLM
cfg=BertConfig(vocab_size=tokenizer.vocab_size, hidden_size=256, num_hidden_layers=4,
               num_attention_heads=4, intermediate_size=1024,
               max_position_embeddings=BLOCK, pad_token_id=tokenizer.pad_token_id)
model=BertForMaskedLM(cfg).to(device)
print("params(M)", round(model.num_parameters()/1e6,2))

## 6. 학습 — plain CE(BertForMaskedLM 기본) + lr 5e-4, 30000 step

In [ ]:
from transformers import Trainer, TrainingArguments
args=TrainingArguments(output_dir="./out34", max_steps=30000,
    per_device_train_batch_size=64, learning_rate=5e-4, weight_decay=0.01,
    warmup_steps=1000, lr_scheduler_type="cosine", max_grad_norm=1.0, fp16=USE_FP16,
    logging_steps=500, save_strategy="no", report_to="none", remove_unused_columns=False, seed=SEED)
trainer=Trainer(model=model, args=args, train_dataset=lm_train, data_collator=coll)
t0=time.time(); r=trainer.train()
print(f"elapsed {(time.time()-t0)/60:.2f}min | step {r.global_step} | train_loss {r.training_loss:.4f} | baseline ln(V) {math.log(tokenizer.vocab_size):.4f}")

## 7. carry-over 샘플러로 한국어 생성

In [ ]:
@torch.no_grad()
def generate(model, length=128, block=32, temperature=0.8, top_p=0.92, top_k=0,
             rep_penalty=1.3, no_immediate_repeat=True, prompt_ids=None):
    """carry-over semi-AR + 반복 억제(rep penalty / 인접중복 금지 / top-p)."""
    model.eval()
    mask_id = tokenizer.mask_token_id
    x = torch.full((1, length), mask_id, dtype=torch.long, device=device)
    fixed = torch.zeros(length, dtype=torch.bool, device=device)
    if prompt_ids is not None:
        p = torch.tensor(prompt_ids[:length], device=device)
        x[0, :len(p)] = p; fixed[:len(p)] = True
    nblocks = (length + block - 1) // block
    for b in range(nblocks):
        lo, hi = b * block, min((b + 1) * block, length)
        steps = hi - lo
        for s in range(steps):
            logits = model(input_ids=x).logits[0].float()        # (L, V)
            logits[:, mask_id] = -1e9
            # 반복 패널티: 이미 확정된 토큰들의 로짓을 깎음
            if rep_penalty and rep_penalty != 1.0:
                comm = x[0][x[0] != mask_id]
                if comm.numel() > 0:
                    u = torch.unique(comm)
                    col = logits[:, u]
                    logits[:, u] = torch.where(col > 0, col / rep_penalty, col * rep_penalty)
            # 인접중복 금지: 각 자리에서 '왼쪽 토큰과 같은 토큰' 예측 차단
            if no_immediate_repeat:
                left = torch.roll(x[0], 1); left[0] = mask_id
                valid = left != mask_id
                logits[valid, left[valid]] = -1e9
            probs = (logits / max(temperature, 1e-6)).softmax(-1)
            if top_k and top_k > 0:
                kth = probs.topk(top_k, dim=-1).values[:, -1, None]
                probs = probs.masked_fill(probs < kth, 0.0)
            if top_p and top_p < 1.0:
                sp, si = probs.sort(dim=-1, descending=True)
                rm = (sp.cumsum(-1) - sp) > top_p
                sp = sp.masked_fill(rm, 0.0)
                probs = torch.zeros_like(probs).scatter(-1, si, sp)
            probs = probs / probs.sum(-1, keepdim=True).clamp_min(1e-9)
            pred = torch.multinomial(probs, 1).squeeze(-1)
            conf = probs.gather(-1, pred.unsqueeze(-1)).squeeze(-1)
            cur = (x[0] == mask_id) & (~fixed)
            cur[:lo] = False; cur[hi:] = False
            nleft = int(cur.sum())
            if nleft == 0: break
            nreveal = nleft if s == steps - 1 else max(1, nleft // (steps - s))
            cc = conf.clone(); cc[~cur] = -1e9
            idx = cc.topk(nreveal).indices
            x[0, idx] = pred[idx]
    return tokenizer.decode(x[0], skip_special_tokens=True)

pid = tokenizer("옛날 옛날에", add_special_tokens=False)["input_ids"]
torch.manual_seed(SEED)
print("=== unconditional (all-[MASK] -> generate, default sampler) ===")
for i in range(3):
    print(f"[{i}] {generate(model)[:340]}")
print("\n=== conditional (prompt 'Once upon a time' fixed) ===")
for i in range(3):
    print(f"[{i}] {generate(model, prompt_ids=pid)[:340]}")

pid = tokenizer("옛날 옛날에", add_special_tokens=False)["input_ids"]
torch.manual_seed(SEED)
print("=== conditional ('옛날 옛날에') ===")
for i in range(3):
    print(f"[{i}] {generate(model, prompt_ids=pid)[:300]}")
print("\n=== unconditional ===")
for i in range(2):
    print(f"[{i}] {generate(model)[:300]}")

## 8. 진단 — 고정-t(0.15) acc + infill

In [ ]:
g=torch.Generator().manual_seed(0)
def fixed_t_acc(tv_=0.15,n=128):
    cor=tot=0
    for ex in lm_val.select(range(min(n,len(lm_val)))):
        ids=torch.tensor(ex["input_ids"]); m=torch.rand(len(ids),generator=g)<tv_
        if not m.any(): m[0]=True
        inp=ids.clone(); inp[m]=tokenizer.mask_token_id
        with torch.no_grad(): pr=model(inp.unsqueeze(0).to(device)).logits[0].argmax(-1).cpu()
        cor+=(pr[m]==ids[m]).sum().item(); tot+=int(m.sum())
    return cor/tot
print(f"[diag] fixed-t(0.15) top-1 acc = {fixed_t_acc():.3f}   (naive diffusion 0.084)")